# 개선 사항 요약 및 결론

## 주요 개선 사항

### 1. 데이터 전처리 개선
- **음수 값 처리**: 절댓값으로 변환하여 매출 취소/오류 데이터 정규화
- **로그 변환**: 매출수량의 왜도를 줄여 모델 학습 안정성 향상
- **이상치 처리**: 극단값이 모델에 미치는 영향 감소

### 2. 피처 엔지니어링
- **시간 기반 피처**: 요일, 월, 계절, 주말/평일 구분
- **이동 통계 피처**: 3일, 7일, 14일 이동평균 및 표준편차
- **Lag 피처**: 1일, 7일 전 매출수량으로 시계열 패턴 반영  
- **영업장 집계 피처**: 영업장별 매출 비중 및 일별 총매출

### 3. 모델 아키텍처 개선
- **Multi-Feature LSTM**: 매출수량 + 추가 피처를 함께 처리
- **Attention Mechanism**: 중요한 시점에 더 많은 가중치 부여
- **Deep Architecture**: 3-layer LSTM으로 복잡한 패턴 학습
- **Dropout**: 정규화로 과적합 방지

### 4. 학습 최적화
- **SMAPE Loss**: 평가지표와 동일한 손실함수로 직접 최적화
- **AdamW Optimizer**: 가중치 감쇠로 일반화 성능 향상
- **Learning Rate Scheduler**: 적응적 학습률 조정
- **Early Stopping**: 과적합 방지 및 최적 모델 선택

### 5. 앙상블 기법
- **가중평균 앙상블**: 베이스라인(30%) + 개선모델(70%)
- **모델 보완**: 개선모델이 예측하지 못한 메뉴는 베이스라인 사용

## 기대 효과

1. **정확도 향상**: 다양한 피처와 attention mechanism으로 예측 정확도 개선
2. **안정성 증가**: 정규화와 드롭아웃으로 모델 일반화 성능 향상  
3. **패턴 인식**: 시간 기반 피처로 계절성/요일별 패턴 반영
4. **평가지표 최적화**: SMAPE 직접 최적화로 대회 성능 향상

## 추가 개선 가능성

1. **외부 정보 활용**: 공휴일, 날씨 등 도메인 지식 기반 피처
2. **교차검증**: 시계열 교차검증으로 모델 성능 검증
3. **하이퍼파라미터 최적화**: Grid Search 또는 Bayesian Optimization
4. **다른 모델 시도**: Transformer, Prophet, LightGBM 등
5. **가중치 조정**: '담하', '미라시아' 영업장에 대한 가중치 반영

In [ ]:
# 최종 예측 및 제출 파일 생성

print("=== 최종 예측 실행 ===")

# 모든 테스트 파일에 대해 예측 수행
all_baseline_preds = []
all_improved_preds = []

test_files = sorted(glob.glob('./test/TEST_*.csv'))
print(f"처리할 테스트 파일 수: {len(test_files)}")

for i, test_file in enumerate(test_files):
    test_df = pd.read_csv(test_file)
    filename = os.path.basename(test_file)
    test_prefix = re.search(r'(TEST_\d+)', filename).group(1)
    
    print(f"처리 중: {test_prefix} ({i+1}/{len(test_files)})")
    
    # 베이스라인 예측
    baseline_pred = predict_lstm(test_df, trained_models, test_prefix)
    all_baseline_preds.append(baseline_pred)
    
    # 개선된 모델 예측 (가능한 경우)
    try:
        if 'improved_models' in globals() and len(improved_models) > 0:
            improved_pred = predict_improved_model(test_df, improved_models, test_prefix)
            all_improved_preds.append(improved_pred)
        else:
            print(f"  - 개선된 모델이 없어 베이스라인 사용")
            all_improved_preds.append(baseline_pred)
    except Exception as e:
        print(f"  - 개선된 모델 예측 실패: {e}")
        all_improved_preds.append(baseline_pred)

# 전체 예측 결과 합치기
full_baseline_pred = pd.concat(all_baseline_preds, ignore_index=True)
full_improved_pred = pd.concat(all_improved_preds, ignore_index=True)

print(f"베이스라인 예측 결과: {len(full_baseline_pred)}개")
print(f"개선된 모델 예측 결과: {len(full_improved_pred)}개")

# 앙상블 예측 생성 (개선된 모델이 있는 경우)
if len(full_improved_pred) > 0 and not full_improved_pred.equals(full_baseline_pred):
    print("앙상블 예측 생성 중...")
    ensemble_pred = create_ensemble_prediction(full_baseline_pred, full_improved_pred, weight_improved=0.7)
    print(f"앙상블 예측 결과: {len(ensemble_pred)}개")
else:
    print("개선된 모델이 없어 베이스라인 사용")
    ensemble_pred = full_baseline_pred

# 최종 제출 파일 생성
print("\n=== 최종 제출 파일 생성 ===")

# 베이스라인 제출 파일
baseline_submission = convert_to_submission_format(full_baseline_pred, sample_submission)
baseline_submission.to_csv('baseline_submission.csv', index=False, encoding='utf-8-sig')
print("베이스라인 제출 파일: baseline_submission.csv")

# 개선된 모델 제출 파일 (앙상블)
improved_submission = convert_to_submission_format(ensemble_pred, sample_submission)
improved_submission.to_csv('improved_submission.csv', index=False, encoding='utf-8-sig')
print("개선된 모델 제출 파일: improved_submission.csv")

# 예측 결과 비교
print("\n=== 예측 결과 비교 ===")
baseline_stats = baseline_submission.iloc[:, 1:].values.flatten()
improved_stats = improved_submission.iloc[:, 1:].values.flatten()

print(f"베이스라인 - 평균: {baseline_stats.mean():.3f}, 표준편차: {baseline_stats.std():.3f}")
print(f"개선 모델 - 평균: {improved_stats.mean():.3f}, 표준편차: {improved_stats.std():.3f}")
print(f"0인 비율 - 베이스라인: {(baseline_stats == 0).mean():.3f}, 개선: {(improved_stats == 0).mean():.3f}")

print("\n=== 제출 파일 검증 ===")
print(f"제출 파일 형태: {improved_submission.shape}")
print(f"샘플 제출 파일 형태: {sample_submission.shape}")
print(f"형태 일치: {improved_submission.shape == sample_submission.shape}")
print(f"컬럼 일치: {list(improved_submission.columns) == list(sample_submission.columns)}")

print("\n최종 제출 파일: improved_submission.csv")

In [ ]:
# 개선된 예측 함수 정의

def predict_improved_model(test_df, improved_models, test_prefix: str):
    """개선된 모델을 사용한 예측"""
    results = []
    
    # 테스트 데이터도 동일한 전처리 적용
    test_processed = preprocess_data(test_df)
    test_processed = create_rolling_features(test_processed)
    test_processed = create_store_features(test_processed)
    
    for store_menu, store_test in test_processed.groupby(['영업장명_메뉴명']):
        if store_menu not in improved_models:
            continue
            
        model_info = improved_models[store_menu]
        model = model_info['model']
        data = model_info['data']
        
        store_test_sorted = store_test.sort_values('영업일자')
        
        # 최근 28일 매출수량 데이터
        recent_sales = store_test_sorted['매출수량'].values[-LOOKBACK:]
        if len(recent_sales) < LOOKBACK:
            continue
            
        # 매출수량 정규화 (학습 시 사용한 scaler 사용)
        recent_sales_scaled = data['sales_scaler'].transform(recent_sales.reshape(-1, 1)).flatten()
        
        # 피처 데이터 (마지막 시점)
        try:
            last_features = store_test_sorted[selected_features].iloc[-1].values
            last_features_scaled = data['feature_scaler'].transform(last_features.reshape(1, -1)).flatten()
        except:
            # 피처가 없는 경우 기본값 사용
            last_features_scaled = np.zeros(len(selected_features))
        
        # 텐서 변환
        X_sales = torch.tensor(recent_sales_scaled.reshape(1, LOOKBACK, 1), dtype=torch.float32).to(DEVICE)
        X_features = torch.tensor(last_features_scaled.reshape(1, -1), dtype=torch.float32).to(DEVICE)
        
        # 예측
        with torch.no_grad():
            pred_scaled = model(X_sales, X_features).squeeze().cpu().numpy()
        
        # 역변환
        restored = []
        for i in range(PREDICT):
            dummy = np.array([[pred_scaled[i]]])
            restored_val = data['sales_scaler'].inverse_transform(dummy)[0, 0]
            restored.append(max(restored_val, 0))  # 음수 → 0
        
        # 예측 결과 저장
        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]
        for d, val in zip(pred_dates, restored):
            results.append({
                '영업일자': d,
                '영업장명_메뉴명': store_menu,
                '매출수량': val
            })
    
    return pd.DataFrame(results)

def create_ensemble_prediction(baseline_pred, improved_pred, weight_improved=0.7):
    """베이스라인과 개선 모델의 앙상블"""
    ensemble_pred = baseline_pred.copy()
    
    # 개선 모델이 예측한 메뉴에 대해서만 앙상블 적용
    improved_dict = dict(zip(
        zip(improved_pred['영업일자'], improved_pred['영업장명_메뉴명']),
        improved_pred['매출수량']
    ))
    
    baseline_dict = dict(zip(
        zip(baseline_pred['영업일자'], baseline_pred['영업장명_메뉴명']),
        baseline_pred['매출수량']
    ))
    
    ensemble_results = []
    for (date, menu), improved_val in improved_dict.items():
        baseline_val = baseline_dict.get((date, menu), 0)
        # 가중평균
        ensemble_val = weight_improved * improved_val + (1 - weight_improved) * baseline_val
        ensemble_results.append({
            '영업일자': date,
            '영업장명_메뉴명': menu,
            '매출수량': ensemble_val
        })
    
    # 개선 모델이 예측하지 못한 메뉴는 베이스라인 사용
    for (date, menu), baseline_val in baseline_dict.items():
        if (date, menu) not in improved_dict:
            ensemble_results.append({
                '영업일자': date,
                '영업장명_메뉴명': menu,
                '매출수량': baseline_val
            })
    
    return pd.DataFrame(ensemble_results)

print("개선된 예측 함수 정의 완료")

In [ ]:
# 멀티피처 데이터 준비 및 모델 학습
try:
    prepared_data = prepare_multi_feature_data(train_processed, selected_features)
    print(f"준비된 메뉴 수: {len(prepared_data)}")
    
    # 모델 학습
    print("개선된 모델 학습 중...")
    improved_models = train_improved_model(prepared_data)
    print(f"학습된 모델 수: {len(improved_models)}")
    
except Exception as e:
    print(f"오류 발생: {e}")
    print("기존 베이스라인 모델을 사용합니다.")

In [ ]:
# 개선된 Multi-Feature LSTM 모델 정의

class ImprovedMultiLSTM(nn.Module):
    def __init__(self, sales_input_dim=1, feature_dim=10, hidden_dim=128, num_layers=3, output_dim=7, dropout=0.2):
        super(ImprovedMultiLSTM, self).__init__()
        
        # 매출수량 시계열을 위한 LSTM
        self.sales_lstm = nn.LSTM(sales_input_dim, hidden_dim//2, num_layers, 
                                 batch_first=True, dropout=dropout)
        
        # 추가 피처를 위한 Dense layer
        self.feature_fc = nn.Linear(feature_dim, hidden_dim//2)
        
        # 통합된 정보를 처리하는 LSTM
        self.combined_lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, 
                                   batch_first=True, dropout=dropout)
        
        # Attention mechanism
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, dropout=dropout)
        
        # Output layers
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim, hidden_dim//2)
        self.fc2 = nn.Linear(hidden_dim//2, output_dim)
        self.relu = nn.ReLU()
        
    def forward(self, sales_seq, features):
        batch_size, seq_len, _ = sales_seq.shape
        
        # 매출수량 시계열 처리
        sales_out, _ = self.sales_lstm(sales_seq)  # (B, T, hidden_dim//2)
        
        # 추가 피처 처리 (각 시점마다 동일한 피처 적용)
        feature_out = self.relu(self.feature_fc(features))  # (B, hidden_dim//2)
        feature_out = feature_out.unsqueeze(1).repeat(1, seq_len, 1)  # (B, T, hidden_dim//2)
        
        # 매출수량과 피처 결합
        combined = torch.cat([sales_out, feature_out], dim=2)  # (B, T, hidden_dim)
        
        # 통합 LSTM
        combined_out, _ = self.combined_lstm(combined)  # (B, T, hidden_dim)
        
        # Attention mechanism
        attn_out, _ = self.attention(combined_out.transpose(0, 1), 
                                   combined_out.transpose(0, 1), 
                                   combined_out.transpose(0, 1))
        attn_out = attn_out.transpose(0, 1)  # (B, T, hidden_dim)
        
        # 최종 출력 (마지막 시점 정보 사용)
        final_hidden = attn_out[:, -1, :]  # (B, hidden_dim)
        
        # Output layers
        out = self.dropout(final_hidden)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        output = self.fc2(out)  # (B, output_dim)
        
        return output

# SMAPE Loss 구현
class SMAPELoss(nn.Module):
    def __init__(self, epsilon=1e-8):
        super(SMAPELoss, self).__init__()
        self.epsilon = epsilon
        
    def forward(self, pred, target):
        # SMAPE = 100 * mean(|pred - target| / (|pred| + |target| + epsilon))
        numerator = torch.abs(pred - target)
        denominator = torch.abs(pred) + torch.abs(target) + self.epsilon
        smape = 100.0 * torch.mean(numerator / denominator)
        return smape

def prepare_multi_feature_data(df, feature_cols):
    """멀티피처 데이터 준비"""
    prepared_data = {}
    
    for store_menu, group in tqdm(df.groupby(['영업장명_메뉴명']), desc='Preparing multi-feature data'):
        store_train = group.sort_values('영업일자').copy()
        
        if len(store_train) < LOOKBACK + PREDICT:
            continue
            
        # 매출수량 정규화
        sales_scaler = MinMaxScaler()
        store_train['매출수량_scaled'] = sales_scaler.fit_transform(store_train[['매출수량']])
        
        # 피처 정규화
        feature_scaler = MinMaxScaler()
        store_train[feature_cols] = feature_scaler.fit_transform(store_train[feature_cols])
        
        # 시계열 시퀀스 생성
        sales_vals = store_train['매출수량_scaled'].values
        feature_vals = store_train[feature_cols].values
        
        X_sales, X_features, y = [], [], []
        for i in range(len(sales_vals) - LOOKBACK - PREDICT + 1):
            # 매출수량 시퀀스
            X_sales.append(sales_vals[i:i+LOOKBACK])
            
            # 피처는 마지막 시점 정보 사용
            X_features.append(feature_vals[i+LOOKBACK-1])
            
            # 타겟
            y.append(sales_vals[i+LOOKBACK:i+LOOKBACK+PREDICT])
        
        if len(X_sales) > 0:
            prepared_data[store_menu] = {
                'X_sales': np.array(X_sales),
                'X_features': np.array(X_features), 
                'y': np.array(y),
                'sales_scaler': sales_scaler,
                'feature_scaler': feature_scaler,
                'feature_cols': feature_cols,
                'last_sales_sequence': sales_vals[-LOOKBACK:],
                'last_features': feature_vals[-1]
            }
    
    return prepared_data

def train_improved_model(prepared_data):
    """개선된 멀티피처 모델 학습"""
    trained_models = {}
    
    # 피처 차원 결정 (첫 번째 데이터에서 추출)
    first_key = list(prepared_data.keys())[0]
    feature_dim = prepared_data[first_key]['X_features'].shape[1]
    
    for store_menu, data in tqdm(prepared_data.items(), desc='Training improved models'):
        
        X_sales = torch.tensor(data['X_sales'], dtype=torch.float32).unsqueeze(-1).to(DEVICE)
        X_features = torch.tensor(data['X_features'], dtype=torch.float32).to(DEVICE)  
        y = torch.tensor(data['y'], dtype=torch.float32).to(DEVICE)
        
        # 모델 생성
        model = ImprovedMultiLSTM(
            sales_input_dim=1,
            feature_dim=feature_dim,
            hidden_dim=128,
            num_layers=3,
            output_dim=PREDICT,
            dropout=0.2
        ).to(DEVICE)
        
        # 옵티마이저 및 손실함수
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
        criterion = SMAPELoss()
        
        model.train()
        best_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(100):  # 더 많은 에포크
            epoch_loss = 0.0
            
            # 배치 학습
            indices = torch.randperm(len(X_sales))
            for i in range(0, len(X_sales), BATCH_SIZE):
                batch_idx = indices[i:i+BATCH_SIZE]
                
                X_sales_batch = X_sales[batch_idx]
                X_features_batch = X_features[batch_idx]
                y_batch = y[batch_idx]
                
                optimizer.zero_grad()
                
                output = model(X_sales_batch, X_features_batch)
                loss = criterion(output, y_batch)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                epoch_loss += loss.item()
            
            avg_loss = epoch_loss / (len(X_sales) // BATCH_SIZE + 1)
            scheduler.step(avg_loss)
            
            # Early stopping
            if avg_loss < best_loss:
                best_loss = avg_loss
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
                
            if patience_counter >= 15:
                break
        
        # 최적 모델 로드
        model.load_state_dict(best_model_state)
        model.eval()
        
        trained_models[store_menu] = {
            'model': model,
            'data': data
        }
    
    return trained_models

print("=== 개선된 Multi-Feature LSTM 모델 ===")

# 주요 피처 선택 (상관관계가 높은 피처들 위주)
selected_features = [
    '요일', '월', '주말', '계절',
    '매출수량_ma_3', '매출수량_ma_7', '매출수량_ma_14',
    '매출수량_lag1', '매출수량_lag7',
    '메뉴_비중'
]

print(f"선택된 피처: {selected_features}")
print("멀티피처 데이터 준비 중...")

In [ ]:
# 개선된 데이터 전처리 및 피처 엔지니어링

def preprocess_data(df):
    """개선된 데이터 전처리 함수"""
    df = df.copy()
    df['영업일자'] = pd.to_datetime(df['영업일자'])
    
    # 1. 음수 값 처리 (절댓값으로 변환)
    df['매출수량'] = df['매출수량'].abs()
    
    # 2. 시간 기반 피처 생성
    df['요일'] = df['영업일자'].dt.dayofweek  # 0: 월요일, 6: 일요일
    df['월'] = df['영업일자'].dt.month
    df['일'] = df['영업일자'].dt.day
    df['주차'] = df['영업일자'].dt.isocalendar().week
    
    # 3. 계절성 피처
    df['계절'] = df['월'].map({12: 0, 1: 0, 2: 0,  # 겨울
                           3: 1, 4: 1, 5: 1,   # 봄
                           6: 2, 7: 2, 8: 2,   # 여름
                           9: 3, 10: 3, 11: 3}) # 가을
    
    # 4. 주말/평일 구분
    df['주말'] = (df['요일'] >= 5).astype(int)
    
    # 5. 영업장 분리
    df['영업장명'] = df['영업장명_메뉴명'].str.split('_').str[0]
    df['메뉴명'] = df['영업장명_메뉴명'].str.split('_', n=1).str[1]
    
    # 6. 로그 변환 (매출수량이 0이 아닌 경우)
    df['매출수량_log'] = np.log1p(df['매출수량'])
    
    return df

def create_rolling_features(df, window_sizes=[3, 7, 14]):
    """이동 통계 피처 생성"""
    df = df.copy()
    df = df.sort_values(['영업장명_메뉴명', '영업일자'])
    
    for window in window_sizes:
        # 이동평균
        df[f'매출수량_ma_{window}'] = df.groupby('영업장명_메뉴명')['매출수량'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
        
        # 이동표준편차
        df[f'매출수량_std_{window}'] = df.groupby('영업장명_메뉴명')['매출수량'].transform(
            lambda x: x.rolling(window, min_periods=1).std().fillna(0)
        )
        
        # 전날 대비 변화율
        if window == 7:
            df['매출수량_pct_change_7d'] = df.groupby('영업장명_메뉴명')['매출수량'].transform(
                lambda x: x.pct_change(periods=7).fillna(0)
            )
    
    # lag 피처 (1일, 7일 전)
    df['매출수량_lag1'] = df.groupby('영업장명_메뉴명')['매출수량'].shift(1).fillna(0)
    df['매출수량_lag7'] = df.groupby('영업장명_메뉴명')['매출수량'].shift(7).fillna(0)
    
    return df

def create_store_features(df):
    """영업장별 집계 피처 생성"""
    df = df.copy()
    
    # 일별 영업장 총 매출
    daily_store_sales = df.groupby(['영업일자', '영업장명'])['매출수량'].sum().reset_index()
    daily_store_sales.rename(columns={'매출수량': '영업장_일매출'}, inplace=True)
    df = df.merge(daily_store_sales, on=['영업일자', '영업장명'], how='left')
    
    # 영업장별 메뉴 비중
    store_menu_ratio = df.groupby(['영업장명', '영업장명_메뉴명'])['매출수량'].sum().reset_index()
    store_total = store_menu_ratio.groupby('영업장명')['매출수량'].sum().reset_index()
    store_total.rename(columns={'매출수량': '영업장_총매출'}, inplace=True)
    store_menu_ratio = store_menu_ratio.merge(store_total, on='영업장명')
    store_menu_ratio['메뉴_비중'] = store_menu_ratio['매출수량'] / store_menu_ratio['영업장_총매출']
    store_menu_ratio = store_menu_ratio[['영업장명_메뉴명', '메뉴_비중']]
    df = df.merge(store_menu_ratio, on='영업장명_메뉴명', how='left')
    
    return df

print("=== 데이터 전처리 및 피처 엔지니어링 ===")

# 원본 데이터 백업
train_original = train.copy()

# 1. 기본 전처리
train_processed = preprocess_data(train)
print(f"1. 기본 전처리 완료 - 새 피처 수: {len(train_processed.columns) - len(train.columns)}")

# 2. 이동 통계 피처 생성
train_processed = create_rolling_features(train_processed)
print(f"2. 이동 통계 피처 생성 완료 - 총 피처 수: {len(train_processed.columns)}")

# 3. 영업장별 집계 피처 생성
train_processed = create_store_features(train_processed)
print(f"3. 영업장별 피처 생성 완료 - 총 피처 수: {len(train_processed.columns)}")

# 피처 목록 출력
feature_cols = [col for col in train_processed.columns if col not in ['영업일자', '영업장명_메뉴명', '영업장명', '메뉴명']]
print(f"\n생성된 피처 목록:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i:2d}. {col}")

# 피처 중요도 분석을 위한 상관관계 확인
numeric_features = train_processed.select_dtypes(include=[np.number]).columns.tolist()
if '매출수량' in numeric_features:
    correlation_with_target = train_processed[numeric_features].corr()['매출수량'].abs().sort_values(ascending=False)
    print(f"\n매출수량과의 상관관계 TOP 10:")
    for i, (feature, corr) in enumerate(correlation_with_target.head(10).items(), 1):
        if feature != '매출수량':
            print(f"{i:2d}. {feature}: {corr:.3f}")

print("\n전처리된 데이터 저장 완료.")

In [ ]:
# 베이스라인 모델 성능 분석 및 개선점 파악

print("=== 베이스라인 모델 분석 ===")
print("현재 베이스라인 모델의 특징:")
print("1. 단순 LSTM 구조 (input_dim=1, hidden_dim=64, num_layers=2)")
print("2. 28일 lookback, 7일 예측")
print("3. MinMaxScaler 정규화")
print("4. MSE Loss 사용")
print("5. 메뉴별 개별 모델 훈련")
print()

print("=== 모델 개선 가능성 분석 ===")

# 1. 데이터 분포 분석
print("1. 데이터 특성 분석:")
zero_ratio = (train['매출수량'] == 0).mean()
negative_ratio = (train['매출수량'] < 0).mean()
positive_data = train[train['매출수량'] > 0]['매출수량']

print(f"   - 0인 데이터 비율: {zero_ratio:.3f} → 희소성 문제")
print(f"   - 음수 데이터 비율: {negative_ratio:.3f} → 전처리 필요")
print(f"   - 양수 데이터 분포: 평균={positive_data.mean():.2f}, 표준편차={positive_data.std():.2f}")
print(f"   - 최대값: {positive_data.max()}")

# 2. 메뉴별 데이터 충분성 분석
menu_data_lengths = train.groupby('영업장명_메뉴명').size()
insufficient_menus = (menu_data_lengths < 35).sum()  # LOOKBACK(28) + PREDICT(7)
print(f"\n2. 학습 데이터 충분성:")
print(f"   - 총 메뉴 수: {len(menu_data_lengths)}")
print(f"   - 학습 불가능 메뉴 수: {insufficient_menus} ({insufficient_menus/len(menu_data_lengths)*100:.1f}%)")
print(f"   - 평균 데이터 길이: {menu_data_lengths.mean():.1f}일")

# 3. 계절성/트렌드 분석
print("\n3. 시계열 패턴 분석:")
# 요일별 패턴
weekday_pattern = train.groupby(train['영업일자'].dt.day_name())['매출수량'].mean()
print(f"   - 요일별 편차: 최대={weekday_pattern.max():.2f}, 최소={weekday_pattern.min():.2f}")

# 월별 패턴  
monthly_pattern = train.groupby(train['영업일자'].dt.month)['매출수량'].mean()
print(f"   - 월별 편차: 최대={monthly_pattern.max():.2f}, 최소={monthly_pattern.min():.2f}")

print("\n=== 개선 방향 ===")
print("1. 데이터 전처리 개선:")
print("   - 음수 값 처리 (절댓값 또는 0으로 변환)")
print("   - 로그 변환으로 분포 정규화")
print("   - 이상치 제거")
print()
print("2. 피처 엔지니어링:")
print("   - 요일, 월, 계절 등 시간 기반 피처")
print("   - 이동평균, 지수평활 피처")
print("   - 영업장별 집계 피처")
print()
print("3. 모델 아키텍처 개선:")
print("   - Multi-feature LSTM (시간 피처 추가)")
print("   - Attention mechanism")
print("   - Ensemble 모델")
print()
print("4. 손실함수 개선:")
print("   - SMAPE를 직접 최적화")
print("   - 가중치 적용 (담하, 미라시아 고려)")
print()
print("5. 후처리:")
print("   - 음수 예측값 → 0으로 변환")
print("   - 메뉴별 최소/최대 제한")

# 베이스라인 모델의 예측 결과 간단 분석
print(f"\n=== 베이스라인 예측 결과 분석 ===")
if os.path.exists('baseline_submission.csv'):
    baseline_pred = pd.read_csv('baseline_submission.csv')
    pred_values = baseline_pred.iloc[:, 1:].values.flatten()
    pred_values = pred_values[pred_values > 0]  # 양수만
    
    print(f"예측값 통계:")
    print(f"   - 평균: {pred_values.mean():.3f}")
    print(f"   - 표준편차: {pred_values.std():.3f}")
    print(f"   - 최대값: {pred_values.max():.3f}")
    print(f"   - 0인 비율: {(baseline_pred.iloc[:, 1:].values == 0).mean():.3f}")
else:
    print("baseline_submission.csv 파일이 없습니다. 먼저 베이스라인 모델을 실행하세요.")

In [ ]:
# 테스트 데이터 분석
print("=== 테스트 데이터 분석 ===")
test_files = sorted(glob.glob('./test/TEST_*.csv'))
print(f"테스트 파일 수: {len(test_files)}")

for test_file in test_files[:2]:  # 처음 2개 파일만 분석
    test_df = pd.read_csv(test_file)
    test_df['영업일자'] = pd.to_datetime(test_df['영업일자'])
    
    print(f"\n{os.path.basename(test_file)}:")
    print(f"  - 데이터 형태: {test_df.shape}")
    print(f"  - 날짜 범위: {test_df['영업일자'].min()} ~ {test_df['영업일자'].max()}")
    print(f"  - 기간: {(test_df['영업일자'].max() - test_df['영업일자'].min()).days + 1}일")
    print(f"  - 고유 메뉴 수: {test_df['영업장명_메뉴명'].nunique()}")
    print(f"  - 매출수량 통계: 평균={test_df['매출수량'].mean():.2f}, 표준편차={test_df['매출수량'].std():.2f}")

# 샘플 제출 파일 분석
print(f"\n=== 샘플 제출 파일 분석 ===")
print(f"제출 형태: {sample_submission.shape}")
print(f"예측해야 할 날짜 수: {len(sample_submission)}")
print(f"예측해야 할 메뉴 수: {len(sample_submission.columns) - 1}")

# 학습 데이터와 테스트 데이터의 메뉴 겹침 확인
train_menus = set(train['영업장명_메뉴명'].unique())
test_menus = set()
for test_file in test_files:
    test_df = pd.read_csv(test_file)
    test_menus.update(test_df['영업장명_메뉴명'].unique())

common_menus = train_menus.intersection(test_menus)
print(f"\n학습 데이터 메뉴 수: {len(train_menus)}")
print(f"테스트 데이터 메뉴 수: {len(test_menus)}")
print(f"공통 메뉴 수: {len(common_menus)}")
print(f"공통 메뉴 비율: {len(common_menus) / len(train_menus):.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 데이터 분석 및 시각화
print("=== 데이터 기본 정보 ===")
print(f"학습 데이터 형태: {train.shape}")
print(f"컬럼: {train.columns.tolist()}")
print(f"데이터 타입: {train.dtypes}")
print()

# 영업장별 분포
print("=== 영업장별 메뉴 수 ===")
store_menu_counts = train['영업장명_메뉴명'].apply(lambda x: x.split('_')[0]).value_counts()
print(store_menu_counts)
print()

# 날짜 범위
print("=== 학습 데이터 날짜 범위 ===")
train['영업일자'] = pd.to_datetime(train['영업일자'])
print(f"시작일: {train['영업일자'].min()}")
print(f"종료일: {train['영업일자'].max()}")
print(f"총 기간: {(train['영업일자'].max() - train['영업일자'].min()).days + 1}일")
print()

# 매출수량 통계
print("=== 매출수량 통계 ===")
print(train['매출수량'].describe())
print(f"0인 비율: {(train['매출수량'] == 0).mean():.3f}")
print(f"음수 비율: {(train['매출수량'] < 0).mean():.3f}")
print()

# 일별 총 매출수량 분포
daily_sales = train.groupby('영업일자')['매출수량'].sum()
print(f"일별 평균 매출수량: {daily_sales.mean():.2f}")
print(f"일별 매출수량 표준편차: {daily_sales.std():.2f}")

# 영업장명_메뉴명 개수
unique_menus = train['영업장명_메뉴명'].nunique()
print(f"고유 메뉴 수: {unique_menus}")

# 시각화 - 매출수량 분포
plt.figure(figsize=(15, 10))

# 1. 매출수량 히스토그램
plt.subplot(2, 3, 1)
plt.hist(train[train['매출수량'] >= 0]['매출수량'], bins=50, alpha=0.7)
plt.title('매출수량 분포 (음수 제외)')
plt.xlabel('매출수량')
plt.ylabel('빈도')

# 2. 일별 총 매출량
plt.subplot(2, 3, 2)
plt.plot(daily_sales.index, daily_sales.values, alpha=0.7)
plt.title('일별 총 매출량')
plt.xlabel('날짜')
plt.ylabel('총 매출수량')
plt.xticks(rotation=45)

# 3. 요일별 매출 패턴
train['요일'] = train['영업일자'].dt.day_name()
weekday_sales = train.groupby('요일')['매출수량'].sum()
weekdays_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
plt.subplot(2, 3, 3)
weekday_sales.reindex(weekdays_order).plot(kind='bar')
plt.title('요일별 총 매출량')
plt.xlabel('요일')
plt.ylabel('총 매출수량')
plt.xticks(rotation=45)

# 4. 월별 매출 패턴
train['월'] = train['영업일자'].dt.month
monthly_sales = train.groupby('월')['매출수량'].sum()
plt.subplot(2, 3, 4)
monthly_sales.plot(kind='bar')
plt.title('월별 총 매출량')
plt.xlabel('월')
plt.ylabel('총 매출수량')

# 5. 영업장별 매출 비중
store_sales = train.groupby(train['영업장명_메뉴명'].str.split('_').str[0])['매출수량'].sum()
plt.subplot(2, 3, 5)
store_sales.plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('영업장별 매출 비중')
plt.ylabel('')

# 6. 매출수량 로그 스케일 분포
plt.subplot(2, 3, 6)
positive_sales = train[train['매출수량'] > 0]['매출수량']
plt.hist(np.log1p(positive_sales), bins=50, alpha=0.7)
plt.title('매출수량 분포 (log scale)')
plt.xlabel('log(매출수량 + 1)')
plt.ylabel('빈도')

plt.tight_layout()
plt.show()

# 상위 매출 메뉴 분석
print("\n=== 상위 매출 메뉴 TOP 10 ===")
menu_sales = train.groupby('영업장명_메뉴명')['매출수량'].sum().sort_values(ascending=False)
print(menu_sales.head(10))

# 매출 패턴 분석을 위한 추가 통계
print("\n=== 매출 패턴 분석 ===")
print(f"매출이 있는 날의 비율: {(daily_sales > 0).mean():.3f}")
print(f"일별 매출 최대값: {daily_sales.max()}")
print(f"일별 매출 최소값: {daily_sales.min()}")

# Import

In [1]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from tqdm import tqdm


# Fixed RandomSeed & Setting Hyperparameter

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data Load

In [12]:
train = pd.read_csv('./train/train.csv')

# Define Model

In [5]:
class MultiOutputLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, output_dim=7):
        super(MultiOutputLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # (B, output_dim)

# Train

In [ ]:
def train_lstm(train_df):
    trained_models = {}

    for store_menu, group in tqdm(train_df.groupby(['영업장명_메뉴명']), desc ='Training LSTM'):
        store_train = group.sort_values('영업일자').copy()
        if len(store_train) < LOOKBACK + PREDICT:
            continue

        features = ['매출수량']
        scaler = MinMaxScaler()
        store_train[features] = scaler.fit_transform(store_train[features])
        train_vals = store_train[features].values  # shape: (N, 1)

        # 시퀀스 구성
        X_train, y_train = [], []
        for i in range(len(train_vals) - LOOKBACK - PREDICT + 1):
            X_train.append(train_vals[i:i+LOOKBACK])
            y_train.append(train_vals[i+LOOKBACK:i+LOOKBACK+PREDICT, 0])

        X_train = torch.tensor(X_train).float().to(DEVICE)
        y_train = torch.tensor(y_train).float().to(DEVICE)

        model = MultiOutputLSTM(input_dim=1, output_dim=PREDICT).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()

        model.train()
        for epoch in range(EPOCHS):
            idx = torch.randperm(len(X_train))
            for i in range(0, len(X_train), BATCH_SIZE):
                batch_idx = idx[i:i+BATCH_SIZE]
                X_batch, y_batch = X_train[batch_idx], y_train[batch_idx]
                output = model(X_batch)
                loss = criterion(output, y_batch)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        trained_models[store_menu] = {
            'model': model.eval(),
            'scaler': scaler,
            'last_sequence': train_vals[-LOOKBACK:]  # (28, 1)
        }

    return trained_models

In [ ]:
# 학습
trained_models = train_lstm(train)

# Prediction

In [8]:
def predict_lstm(test_df, trained_models, test_prefix: str):
    results = []

    for store_menu, store_test in test_df.groupby(['영업장명_메뉴명']):
        key = store_menu
        if key not in trained_models:
            continue

        model = trained_models[key]['model']
        scaler = trained_models[key]['scaler']

        store_test_sorted = store_test.sort_values('영업일자')
        recent_vals = store_test_sorted['매출수량'].values[-LOOKBACK:]
        if len(recent_vals) < LOOKBACK:
            continue

        # 정규화
        recent_vals = scaler.transform(recent_vals.reshape(-1, 1))
        x_input = torch.tensor([recent_vals]).float().to(DEVICE)

        with torch.no_grad():
            pred_scaled = model(x_input).squeeze().cpu().numpy()

        # 역변환
        restored = []
        for i in range(PREDICT):
            dummy = np.zeros((1, 1))
            dummy[0, 0] = pred_scaled[i]
            restored_val = scaler.inverse_transform(dummy)[0, 0]
            restored.append(max(restored_val, 0))

        # 예측일자: TEST_00+1일 ~ TEST_00+7일
        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]

        for d, val in zip(pred_dates, restored):
            results.append({
                '영업일자': d,
                '영업장명_메뉴명': store_menu,
                '매출수량': val
            })

    return pd.DataFrame(results)


In [ ]:
all_preds = []

# 모든 test_*.csv 순회
test_files = sorted(glob.glob('./test/TEST_*.csv'))

for path in test_files:
    test_df = pd.read_csv(path)

    # 파일명에서 접두어 추출 (예: TEST_00)
    filename = os.path.basename(path)
    test_prefix = re.search(r'(TEST_\d+)', filename).group(1)

    pred_df = predict_lstm(test_df, trained_models, test_prefix)
    all_preds.append(pred_df)
    
full_pred_df = pd.concat(all_preds, ignore_index=True)

# Submission

In [ ]:
def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    # (영업일자, 메뉴) → 매출수량 딕셔너리로 변환
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']),
        pred_df['매출수량']
    ))

    final_df = sample_submission.copy()

    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:  # 메뉴명들
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)

    return final_df

In [14]:
sample_submission = pd.read_csv('./sample_submission.csv')
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.to_csv('baseline_submission.csv', index=False, encoding='utf-8-sig')